# OOP Castle Architecture · Dimensional Analysis · Flood Simulation · Torch Room Algebra
**File:** `oop_castle_flood_001.ipynb`

**Stack:** Python OOP → SymPy (calculus + dimensional analysis) → NumPy (geometry + FTCS flood) → Torch (batch algebra)

§1 Class hierarchy: Material → Surface → Wall/Door/Window → Room → Floor → Building  
§2 Dimensional analysis — Buckingham π, SymPy unit checking for architecture & fluids  
§3 Calculus — area integrals, centroid, moment of inertia, room acoustics Sabine formula  
§4 Interior geometry — ray casting, farthest visible point, convex hull, sight lines  
§5 Flood simulation — 2D shallow-water FTCS on room footprint, door as open boundary  
§6 Torch room algebra — batch affine transforms, wall collisions, space metrics

In [1]:
import sympy as sp
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict
from IPython.display import display, Math
import math, copy, warnings; warnings.filterwarnings('ignore')

sp.init_printing(use_latex='mathjax')
plt.rcParams.update({'font.size': 10, 'figure.dpi': 110})
torch.set_default_dtype(torch.float64)
print('ready')

ready


---
# §1 — OOP Class Hierarchy: Material → Surface → Room → Building

**Design principle (bottom-up, like Verilog):**
```
Material        ← thermal k, density, cost/m²
  └─ Surface     ← area, material, orientation normal
       ├─ Wall        ← two endpoints, height, openings list
       ├─ Floor       ← polygon vertices, load capacity
       ├─ Ceiling     ← polygon vertices, height
       ├─ Door        ← width, height, swing angle, Wall parent
       └─ Window      ← width, height, sill height, Wall parent
Room            ← walls list, floor, ceiling, name, function
  └─ Floor (building floor)  ← rooms list, elevation
       └─ Building    ← floors list, structural system, location
```

In [2]:
# ════════════════════════════════════════════════════════
# MATERIAL
# ════════════════════════════════════════════════════════
@dataclass
class Material:
    name:         str
    thermal_k:    float   # W/(m·K)  thermal conductivity
    density:      float   # kg/m³
    cost_per_m2:  float   # USD/m²
    r_value:      float   # m²·K/W   thermal resistance per unit thickness
    absorb_coeff: float   # 0–1 acoustic absorption @ 1 kHz

    def u_value(self, thickness_m: float) -> float:
        """U-value = 1/(R*thickness) [W/(m²·K)]"""
        return 1.0 / (self.r_value * thickness_m + 1e-9)

    def heat_flux(self, thickness_m: float, delta_T: float) -> float:
        """q = k*ΔT/d  [W/m²]"""
        return self.thermal_k * delta_T / thickness_m

    def __repr__(self):
        return f'Material({self.name}, k={self.thermal_k}, ρ={self.density})'


# ════════════════════════════════════════════════════════
# SURFACE (abstract base)
# ════════════════════════════════════════════════════════
class Surface:
    def __init__(self, material: Material, thickness: float):
        self.material  = material
        self.thickness = thickness    # meters

    @property
    def area(self) -> float:
        raise NotImplementedError

    @property
    def mass(self) -> float:
        return self.area * self.thickness * self.material.density

    @property
    def thermal_resistance(self) -> float:
        return self.material.r_value * self.thickness

    @property
    def cost(self) -> float:
        return self.area * self.material.cost_per_m2

    def __repr__(self):
        return f'{type(self).__name__}(A={self.area:.2f}m², mat={self.material.name})'


# ════════════════════════════════════════════════════════
# OPENING (Door / Window — embedded in Wall)
# ════════════════════════════════════════════════════════
class Opening(Surface):
    def __init__(self, width: float, height: float,
                 sill_height: float, material: Material, thickness: float):
        super().__init__(material, thickness)
        self.width       = width
        self.height      = height
        self.sill_height = sill_height   # from floor

    @property
    def area(self) -> float:
        return self.width * self.height


class Door(Opening):
    def __init__(self, width: float, height: float, material: Material,
                 swing_inward: bool = True, thickness: float = 0.045):
        super().__init__(width, height, 0.0, material, thickness)
        self.swing_inward = swing_inward
        self.is_open      = False
        self.open_angle   = 0.0   # radians

    def open(self, angle_deg: float = 90.0):
        self.is_open    = True
        self.open_angle = math.radians(angle_deg)

    def close(self):
        self.is_open    = False
        self.open_angle = 0.0

    def hydraulic_diameter(self) -> float:
        """D_h = 4A/P — used in flood/flow calculations."""
        P = 2*(self.width + self.height)
        return 4*self.area / P

    def __repr__(self):
        state = f'open={self.open_angle*180/math.pi:.0f}°' if self.is_open else 'closed'
        return f'Door({self.width:.2f}×{self.height:.2f}m, {state})'


class Window(Opening):
    def __init__(self, width: float, height: float,
                 sill_height: float, material: Material,
                 u_factor: float = 1.4, vt: float = 0.6):
        super().__init__(width, height, sill_height, material, 0.025)
        self.u_factor = u_factor    # W/(m²·K)
        self.vt       = vt          # visible transmittance 0–1

    def solar_gain(self, irradiance: float, shgc: float = 0.4) -> float:
        """Solar heat gain [W] = SHGC * A * irradiance"""
        return shgc * self.area * irradiance

    def __repr__(self):
        return f'Window({self.width:.2f}×{self.height:.2f}m, sill={self.sill_height:.2f}m)'


# ════════════════════════════════════════════════════════
# WALL
# ════════════════════════════════════════════════════════
class Wall(Surface):
    def __init__(self, p0: Tuple, p1: Tuple,
                 height: float, material: Material, thickness: float = 0.2):
        super().__init__(material, thickness)
        self.p0       = np.array(p0, dtype=float)   # (x, y) in floor plan
        self.p1       = np.array(p1, dtype=float)
        self.height   = height
        self.openings: List[Opening] = []

    @property
    def length(self) -> float:
        return float(np.linalg.norm(self.p1 - self.p0))

    @property
    def gross_area(self) -> float:
        return self.length * self.height

    @property
    def area(self) -> float:
        return self.gross_area - sum(o.area for o in self.openings)

    @property
    def normal(self) -> np.ndarray:
        """Outward normal (2D, perpendicular to wall direction)."""
        d = self.p1 - self.p0
        n = np.array([-d[1], d[0]]) / self.length
        return n

    def add_opening(self, opening: Opening):
        self.openings.append(opening)

    def midpoint(self) -> np.ndarray:
        return (self.p0 + self.p1) / 2

    def heat_loss(self, delta_T: float) -> float:
        """Q = A * U * ΔT  [W]"""
        wall_loss = self.area * self.material.u_value(self.thickness) * delta_T
        opening_loss = sum(o.material.u_value(o.thickness) * o.area * delta_T
                           for o in self.openings)
        return wall_loss + opening_loss

    def __repr__(self):
        return (f'Wall(({self.p0[0]:.1f},{self.p0[1]:.1f})->'
                f'({self.p1[0]:.1f},{self.p1[1]:.1f}), '
                f'L={self.length:.2f}m, {len(self.openings)} openings)')


# ════════════════════════════════════════════════════════
# FLOOR SLAB & CEILING
# ════════════════════════════════════════════════════════
class FloorSlab(Surface):
    def __init__(self, vertices: List[Tuple], material: Material, thickness: float = 0.2):
        super().__init__(material, thickness)
        self.vertices = [np.array(v, dtype=float) for v in vertices]

    @property
    def area(self) -> float:
        """Shoelace formula for polygon area."""
        pts = self.vertices
        n = len(pts)
        A = 0.0
        for i in range(n):
            j = (i+1) % n
            A += pts[i][0]*pts[j][1] - pts[j][0]*pts[i][1]
        return abs(A) / 2

    @property
    def centroid(self) -> np.ndarray:
        pts = self.vertices; n = len(pts)
        cx = cy = 0.0; A = self.area
        for i in range(n):
            j = (i+1) % n
            cross = pts[i][0]*pts[j][1] - pts[j][0]*pts[i][1]
            cx += (pts[i][0]+pts[j][0]) * cross
            cy += (pts[i][1]+pts[j][1]) * cross
        return np.array([cx/(6*A), cy/(6*A)])

    def moment_of_inertia(self) -> Tuple[float,float,float]:
        """Second moments of area Ixx, Iyy, Ixy about centroid."""
        pts = self.vertices; n = len(pts); cx, cy = self.centroid
        Ixx = Iyy = Ixy = 0.0
        for i in range(n):
            j = (i+1) % n
            xi, yi = pts[i] - np.array([cx,cy])
            xj, yj = pts[j] - np.array([cx,cy])
            cross = xi*yj - xj*yi
            Ixx += (yi**2 + yi*yj + yj**2) * cross
            Iyy += (xi**2 + xi*xj + xj**2) * cross
            Ixy += (xi*yj + 2*xi*yi + 2*xj*yj + xj*yi) * cross
        return abs(Ixx)/12, abs(Iyy)/12, Ixy/24


class Ceiling(FloorSlab):
    pass  # same geometry, different semantics


# ════════════════════════════════════════════════════════
# ROOM
# ════════════════════════════════════════════════════════
class Room:
    def __init__(self, name: str, function: str,
                 vertices: List[Tuple], height: float,
                 wall_material: Material, floor_material: Material):
        self.name     = name
        self.function = function
        self.height   = height
        self.floor    = FloorSlab(vertices, floor_material)
        self.ceiling  = Ceiling(vertices, floor_material)
        self.walls: List[Wall] = []
        # Build walls from polygon edges
        verts = self.floor.vertices
        for i in range(len(verts)):
            j = (i+1) % len(verts)
            w = Wall(tuple(verts[i]), tuple(verts[j]),
                     height, wall_material)
            self.walls.append(w)

    # ── area / volume ──────────────────────────────────────────────
    @property
    def floor_area(self) -> float:   return self.floor.area

    @property
    def volume(self) -> float:       return self.floor_area * self.height

    @property
    def total_wall_area(self) -> float:
        return sum(w.area for w in self.walls)

    @property
    def surface_area(self) -> float:
        return self.total_wall_area + 2*self.floor_area  # walls + floor + ceiling

    # ── acoustics: Sabine reverberation time ──────────────────────
    def sabine_RT60(self) -> float:
        """
        RT60 = 0.161 * V / (sum α_i * A_i)  [seconds]
        Sabine formula: time for sound to decay 60 dB.
        """
        S_abs = sum(w.area * w.material.absorb_coeff for w in self.walls)
        S_abs += self.floor.area * self.floor.material.absorb_coeff
        S_abs += self.ceiling.area * self.ceiling.material.absorb_coeff
        if S_abs < 1e-9: return float('inf')
        return 0.161 * self.volume / S_abs

    # ── thermal: total heat loss ───────────────────────────────────
    def heat_loss(self, delta_T: float) -> float:
        return sum(w.heat_loss(delta_T) for w in self.walls)

    # ── centroid ───────────────────────────────────────────────────
    @property
    def centroid(self) -> np.ndarray:
        return self.floor.centroid

    # ── add door to a wall ─────────────────────────────────────────
    def add_door(self, wall_idx: int, door: Door):
        self.walls[wall_idx].add_opening(door)

    def add_window(self, wall_idx: int, window: Window):
        self.walls[wall_idx].add_opening(window)

    def report(self):
        print(f'Room: {self.name} ({self.function})')
        print(f'  Floor area:   {self.floor_area:.2f} m²')
        print(f'  Volume:       {self.volume:.2f} m³')
        print(f'  Wall area:    {self.total_wall_area:.2f} m²')
        print(f'  RT60 (Sabine):{self.sabine_RT60():.2f} s')
        print(f'  Heat loss ΔT=20K: {self.heat_loss(20):.1f} W')
        print(f'  Centroid:     ({self.centroid[0]:.2f}, {self.centroid[1]:.2f}) m')

    def __repr__(self):
        return f'Room("{self.name}", A={self.floor_area:.1f}m², V={self.volume:.1f}m³)'


# ════════════════════════════════════════════════════════
# BUILDING FLOOR (storey)
# ════════════════════════════════════════════════════════
class BuildingFloor:
    def __init__(self, number: int, elevation: float):
        self.number    = number
        self.elevation = elevation   # meters above grade
        self.rooms: List[Room] = []

    def add_room(self, room: Room): self.rooms.append(room)

    @property
    def total_area(self): return sum(r.floor_area for r in self.rooms)

    def __repr__(self):
        return f'Floor({self.number}, z={self.elevation}m, rooms={len(self.rooms)}, A={self.total_area:.1f}m²)'


# ════════════════════════════════════════════════════════
# BUILDING
# ════════════════════════════════════════════════════════
class Building:
    def __init__(self, name: str, location: str, structural_system: str):
        self.name              = name
        self.location          = location
        self.structural_system = structural_system
        self.floors: List[BuildingFloor] = []

    def add_floor(self, floor: BuildingFloor): self.floors.append(floor)

    @property
    def total_area(self): return sum(f.total_area for f in self.floors)

    @property
    def n_rooms(self): return sum(len(f.rooms) for f in self.floors)

    def all_rooms(self) -> List[Room]:
        return [r for f in self.floors for r in f.rooms]

    def report(self):
        print(f'Building: {self.name} @ {self.location}')
        print(f'  Structural: {self.structural_system}')
        print(f'  Floors: {len(self.floors)}, Rooms: {self.n_rooms}')
        print(f'  Total area: {self.total_area:.1f} m²')
        for fl in self.floors:
            print(f'    {fl}')
            for r in fl.rooms:
                print(f'      {r}')

    def __repr__(self):
        return f'Building("{self.name}", {len(self.floors)} floors, {self.total_area:.1f}m²)'


In [5]:
# ── Instantiate a castle wing ─────────────────────────────────────
# Materials
# Accept legacy 'k' keyword for Material (backwards compatibility)
_orig_Material_init = Material.__init__
def _Material_init_with_k(self, *args, **kwargs):
    if 'k' in kwargs:
        kwargs['thermal_k'] = kwargs.pop('k')
    return _orig_Material_init(self, *args, **kwargs)
Material.__init__ = _Material_init_with_k

stone = Material('stone', thermal_k=2.0, density=2400, cost_per_m2=180, r_value=0.6, absorb_coeff=0.02)
wood      = Material('wood',        k=0.13,density=600,  cost_per_m2=80,  r_value=4.5, absorb_coeff=0.15)
oak_door  = Material('oak',         k=0.17,density=700,  cost_per_m2=300, r_value=3.8, absorb_coeff=0.10)
glass     = Material('glass',       k=1.0, density=2500, cost_per_m2=120, r_value=0.18,absorb_coeff=0.03)
plaster   = Material('plaster',     k=0.57,density=1200, cost_per_m2=40,  r_value=1.1, absorb_coeff=0.08)
tapestry  = Material('tapestry',    k=0.05,density=200,  cost_per_m2=500, r_value=6.5, absorb_coeff=0.65)

# Castle rooms
rooms_def = [
    ('Great Hall',     'assembly',   [(0,0),(12,0),(12,8),(0,8)],        6.0, stone,   stone),
    ('Throne Room',    'ceremonial', [(0,8),(10,8),(10,15),(0,15)],       5.0, stone,   stone),
    ('Armoury',        'storage',    [(12,0),(18,0),(18,6),(12,6)],       4.0, stone,   stone),
    ('Kitchen',        'service',    [(12,6),(20,6),(20,14),(12,14)],     3.5, plaster, wood),
    ('Chapel',         'sacred',     [(0,15),(7,15),(7,22),(0,22)],       8.0, stone,   stone),
    ('Lords Chamber',  'private',    [(7,15),(14,15),(14,22),(7,22)],     4.0, tapestry,wood),
    ('Dungeon',        'prison',     [(0,-6),(10,-6),(10,0),(0,0)],       3.0, stone,   stone),
    ('Tower Room',     'observation',[(14,18),(18,18),(18,22),(14,22)],  10.0, stone,   stone),
]

castle = Building('Castle Ravenskeep', 'Northern Scotland', 'load-bearing masonry')
ground_floor = BuildingFloor(1, 0.0)
upper_floor  = BuildingFloor(2, 4.0)

room_objects = {}
for name, func, verts, ht, wmat, fmat in rooms_def:   # loop: build all rooms
    r = Room(name, func, verts, ht, wmat, fmat)
    room_objects[name] = r
    if 'Tower' in name or 'Lords' in name:
        upper_floor.add_room(r)
    else:
        ground_floor.add_room(r)

castle.add_floor(ground_floor)
castle.add_floor(upper_floor)

# Add doors
main_door   = Door(1.2, 2.4, oak_door, swing_inward=True)
inner_door  = Door(0.9, 2.1, oak_door, swing_inward=True)
chapel_door = Door(1.5, 3.0, oak_door, swing_inward=False)

room_objects['Great Hall'].add_door(0, main_door)
room_objects['Great Hall'].add_door(2, inner_door)
room_objects['Chapel'].add_door(0, chapel_door)

# Add windows (Great Hall — high narrow arrow loops)
for wi, wall_idx in enumerate([0, 1, 2, 3]):      # loop: add windows
    w = Window(0.3, 1.2, 3.0, glass)
    room_objects['Great Hall'].add_window(wall_idx, w)

castle.report()

Building: Castle Ravenskeep @ Northern Scotland
  Structural: load-bearing masonry
  Floors: 2, Rooms: 8
  Total area: 440.0 m²
    Floor(1, z=0.0m, rooms=6, A=375.0m²)
      Room("Great Hall", A=96.0m², V=576.0m³)
      Room("Throne Room", A=70.0m², V=350.0m³)
      Room("Armoury", A=36.0m², V=144.0m³)
      Room("Kitchen", A=64.0m², V=224.0m³)
      Room("Chapel", A=49.0m², V=392.0m³)
      Room("Dungeon", A=60.0m², V=180.0m³)
    Floor(2, z=4.0m, rooms=2, A=65.0m²)
      Room("Lords Chamber", A=49.0m², V=196.0m³)
      Room("Tower Room", A=16.0m², V=160.0m³)


In [ ]:
# ── Loop over all rooms: analysis table ──────────────────────────
print(f'{"Room":18} {"Area m²":8} {"Vol m³":8} {"RT60 s":8} {"HeatLoss W":10} {"Cost $":8}')
print('='*70)
castle_obj = globals().get('castle')
if castle_obj is None:
    print("Error: 'castle' is not defined. Run the cell that builds the castle (cell 4).")
else:
    for room in castle_obj.all_rooms():                   # loop over rooms
        cost = sum(w.cost for w in room.walls) + room.floor.cost
        print(f'{room.name:18} {room.floor_area:8.1f} {room.volume:8.1f} '
              f'{room.sabine_RT60():8.2f} {room.heat_loss(20):10.0f} {cost:8.0f}')

# ── Floor plan visualization ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 10))
cmap = cm.Set3
for i, room in enumerate(castle.all_rooms()):     # loop: draw rooms
    pts = np.array([v for v in room.floor.vertices])
    poly = plt.Polygon(pts, closed=True, alpha=0.4,
                       facecolor=cmap(i/len(castle.all_rooms())), edgecolor='k', lw=2)
    ax.add_patch(poly)
    cx, cy = room.centroid
    ax.text(cx, cy, f'{room.name}\n{room.floor_area:.0f}m²',
            ha='center', va='center', fontsize=8, fontweight='bold')
    # draw openings on walls
    for wall in room.walls:
        for opening in wall.openings:
            mp = wall.midpoint()
            sym = 'D' if isinstance(opening, Door) else 'W'
            color = 'brown' if isinstance(opening, Door) else 'cyan'
            ax.plot(mp[0], mp[1], 's', color=color, ms=8, zorder=5)
            ax.text(mp[0]+0.3, mp[1]+0.3, sym, fontsize=7, color=color)

ax.set_xlim(-2, 22); ax.set_ylim(-8, 24)
ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('Castle Ravenskeep — Floor Plan (D=Door, W=Window)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

Room               Area m²  Vol m³   RT60 s   HeatLoss W Cost $  


NameError: name 'castle' is not defined

---
# §2 — Dimensional Analysis: Buckingham π Theorem + SymPy Unit Checking

**Buckingham π:** if a problem has $n$ variables and $k$ independent dimensions,  
there are $n - k$ dimensionless groups $\\Pi_i$.  
Used to derive scaling laws without solving the full PDE — critical for flood modeling.

In [ ]:
# Dimensions: M (mass), L (length), T (time), Θ (temperature)
# [M, L, T, Θ]

# ── Problem 1: Shallow water flood in a room ──────────────────────
# Variables: h (depth), v (velocity), g (gravity), ν (kinematic viscosity),
#            L_room (length), t (time), ρ (density)
# Dimensions matrix [rows=dims, cols=variables]
var_names = ['h', 'v', 'g', 'nu',  'L', 't',  'rho']
#              L   L/T  L/T² L²/T   L    T    M/L³
dim_matrix = np.array([
#              M   L    T    Θ
    [0,        1,  0,   0,  0,   0,   1],   # M
    [0,        1,  1,   2,  1,   0,  -3],   # L
    [0,       -1, -2,  -1,  0,   1,   0],   # T
], dtype=float)

# rank of dimension matrix
rank_k = np.linalg.matrix_rank(dim_matrix)
n_vars = len(var_names)
n_pi   = n_vars - rank_k
print(f'Flood variables: n={n_vars}, rank(k)={rank_k}, π groups = {n_pi}')

# ── SymPy: derive classic dimensionless groups ─────────────────────
# Define physical dimensions symbolically
M, L, T_dim, Theta = sp.symbols('M L T Theta', positive=True)

dimensions = {
    'h':          L,
    'v':          L/T_dim,
    'g':          L/T_dim**2,
    'nu':         L**2/T_dim,
    'L_room':     L,
    'rho':        M/L**3,
    'mu':         M/(L*T_dim),
    'sigma':      M/T_dim**2,           # surface tension
    'P':          M/(L*T_dim**2),       # pressure
    'k_thermal':  M*L/(T_dim**3*Theta), # thermal conductivity
    'alpha':      L**2/T_dim,           # thermal diffusivity
    'c_p':        L**2/(T_dim**2*Theta),# specific heat
    'DeltaT':     Theta,
}

# Classic π groups for architecture / flood
pi_groups = [
    ('Froude  Fr', 'v / sqrt(g*h)',        'inertia/gravity — flood wave speed'),
    ('Reynolds Re','rho*v*L_room/mu',       'inertia/viscosity — turbulence'),
    ('Weber   We', 'rho*v^2*L_room/sigma', 'inertia/surface tension'),
    ('Nusselt Nu', 'h_conv*L/k_thermal',   'convective/conductive heat transfer'),
    ('Prandtl Pr', 'nu/alpha',             'momentum/thermal diffusivity'),
    ('Biot    Bi', 'h_conv*L/k_solid',     'surface/internal conduction'),
    ('Aspect  AR', 'L_room/h_room',        'room geometry — width to height'),
    ('Acoust  AR', 'RT60*c_sound/V^(1/3)', 'reverberation / room size'),
]

print('\nDimensionless groups (π) for architectural flood physics:')
print(f'{"Group":15}  {"Formula":35}  Physical meaning')
print('='*80)
for name, formula, meaning in pi_groups:   # loop over π groups
    print(f'{name:15}  {formula:35}  {meaning}')

# ── Numerical: compute Froude and Reynolds for flood scenario ─────
g_val = 9.81   # m/s²
rho_w = 1000   # kg/m³
mu_w  = 1e-3   # Pa·s (water at 20°C)
nu_w  = mu_w / rho_w

print('\nFlood scenario analysis for Great Hall (12m × 8m):')
h_depths = np.array([0.05, 0.10, 0.20, 0.50, 1.0])  # water depth [m]
L_room   = 12.0
print(f'{"h (m)":8}  {"v_wave m/s":12}  {"Fr":8}  {"Re":12}  {"Flow regime"}')
print('-'*60)
for h in h_depths:              # loop over flood depths
    v_wave = np.sqrt(g_val * h)  # shallow-water wave speed
    Fr     = v_wave / np.sqrt(g_val * h)
    Re     = rho_w * v_wave * L_room / mu_w
    regime = 'subcritical' if Fr < 1 else 'supercritical'
    turb   = 'turbulent' if Re > 4000 else 'laminar'
    print(f'{h:8.2f}  {v_wave:12.3f}  {Fr:8.3f}  {Re:12.0f}  {regime}, {turb}')

---
# §3 — Calculus: Area Integrals, Centroid, Moment of Inertia, Sabine

**SymPy** computes exact integrals for arbitrary room polygons.  
The second moments of area ($I_{xx}$, $I_{yy}$) determine structural stiffness and flood pressure distribution.

In [ ]:
# ── SymPy: centroid and Ixx integrals for a general rectangle ────
x_s, y_s, W_s, H_s = sp.symbols('x y W H', positive=True)

# Area of rectangle W×H
A_rect = sp.integrate(sp.integrate(1, (y_s, 0, H_s)), (x_s, 0, W_s))
print('Area of W×H rectangle:'); display(Math(sp.latex(sp.Eq(sp.Symbol('A'), A_rect))))

# Centroid x̄
x_bar = sp.integrate(sp.integrate(x_s, (y_s, 0, H_s)), (x_s, 0, W_s)) / A_rect
y_bar = sp.integrate(sp.integrate(y_s, (y_s, 0, H_s)), (x_s, 0, W_s)) / A_rect
print('Centroid:')
display(Math(r'\bar{x} = ' + sp.latex(sp.simplify(x_bar)) + r',\quad \bar{y} = ' + sp.latex(sp.simplify(y_bar))))

# Second moments about centroid
x_c, y_c = x_s - W_s/2, y_s - H_s/2
Ixx_sym = sp.integrate(sp.integrate(y_c**2, (y_s, 0, H_s)), (x_s, 0, W_s))
Iyy_sym = sp.integrate(sp.integrate(x_c**2, (y_s, 0, H_s)), (x_s, 0, W_s))
print('Second moments of area about centroid:')
display(Math(r'I_{xx} = ' + sp.latex(sp.simplify(Ixx_sym)) +
             r',\quad I_{yy} = ' + sp.latex(sp.simplify(Iyy_sym))))

# ── Numerical: loop over all castle rooms ────────────────────────
print('\nRoom geometry analysis (shoelace + moment integrals):')
print(f'{"Room":18}  {"A m²":7}  {"Cx":6}  {"Cy":6}  {"Ixx":10}  {"Iyy":10}')
print('-'*70)
for room in castle.all_rooms():            # loop over rooms
    Ixx, Iyy, Ixy = room.floor.moment_of_inertia()
    cx, cy = room.centroid
    print(f'{room.name:18}  {room.floor_area:7.2f}  {cx:6.2f}  {cy:6.2f}  {Ixx:10.2f}  {Iyy:10.2f}')

# ── Sabine formula: SymPy symbolic ───────────────────────────────
V_s, alpha_s, S_s, c_s = sp.symbols('V alpha_bar S c', positive=True)
RT60_sym = sp.Rational(24) * sp.log(10) / c_s * V_s / (alpha_s * S_s)
RT60_approx = sp.Rational(161, 1000) * V_s / (alpha_s * S_s)
print('\nSabine reverberation time (exact vs approximation):')
display(Math(r'T_{60} = \frac{24\ln 10}{c} \frac{V}{\bar{\alpha}S} \approx \frac{0.161 V}{\bar{\alpha}S}'))

# Loop over rooms: compare RT60 to acoustic targets
targets = {
    'assembly': (1.2, 2.5), 'ceremonial': (1.5, 3.0), 'storage': (0.5, 2.0),
    'service':  (0.4, 1.0), 'sacred':     (2.0, 4.0), 'private': (0.4, 0.8),
    'prison':   (0.5, 2.0), 'observation':(0.5, 1.5),
}
print('\nAcoustic assessment:')
print(f'{"Room":18}  {"RT60 s":8}  {"Target":12}  Status')
print('-'*55)
for room in castle.all_rooms():            # loop over rooms
    rt = room.sabine_RT60()
    lo, hi = targets.get(room.function, (0.5, 2.0))
    ok = 'GOOD' if lo <= rt <= hi else ('TOO LIVE' if rt > hi else 'TOO DEAD')
    print(f'{room.name:18}  {rt:8.2f}  [{lo:.1f} – {hi:.1f}]    {ok}')

---
# §4 — Interior Geometry: Ray Casting, Farthest Point, Sight Lines

**Farthest visible point:** from any observer position inside a room, cast rays at all angles  
and find the intersection with walls. The farthest ray gives maximum line-of-sight distance.

In [ ]:
def ray_wall_intersect(ray_origin, ray_dir, wall_p0, wall_p1):
    """
    Compute intersection of ray (origin + t*dir) with line segment (p0->p1).
    Returns t >= 0 if hit, else None.
    """
    d  = ray_dir
    e  = wall_p1 - wall_p0
    denom = d[0]*e[1] - d[1]*e[0]
    if abs(denom) < 1e-10: return None
    f = wall_p0 - ray_origin
    t = (f[0]*e[1] - f[1]*e[0]) / denom
    u = (f[0]*d[1] - f[1]*d[0]) / denom
    if t >= 0 and 0 <= u <= 1:
        return t
    return None


def cast_rays(observer, room, n_rays=360):
    """
    Cast n_rays from observer, return (angles, distances, hit_points).
    Loop over angles, then over walls for each ray.
    """
    angles = np.linspace(0, 2*np.pi, n_rays, endpoint=False)
    dists  = np.zeros(n_rays)
    hits   = np.zeros((n_rays, 2))

    for i, angle in enumerate(angles):          # loop over ray angles
        ray_dir = np.array([np.cos(angle), np.sin(angle)])
        t_min = np.inf
        for wall in room.walls:                  # loop over walls
            t = ray_wall_intersect(observer, ray_dir, wall.p0, wall.p1)
            if t is not None and t < t_min:
                t_min = t
        dists[i] = t_min if t_min < np.inf else 0
        hits[i]  = observer + t_min * ray_dir if t_min < np.inf else observer
    return angles, dists, hits


def farthest_point_from(observer, room, n_rays=1440):
    """Returns (angle, distance, point) of farthest visible wall."""
    angles, dists, hits = cast_rays(observer, room, n_rays)
    idx = np.argmax(dists)
    return angles[idx], dists[idx], hits[idx]


# ── visualize for Great Hall and Chapel ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, room_name in zip(axes, ['Great Hall', 'Chapel']):
    room = room_objects[room_name]
    pts = np.array([v for v in room.floor.vertices])
    poly = plt.Polygon(pts, closed=True, alpha=0.15, facecolor='tan', edgecolor='k', lw=2)
    ax.add_patch(poly)

    # observer at centroid
    obs = room.centroid

    # cast 720 rays
    angles, dists, hits = cast_rays(obs, room, n_rays=720)

    # draw every 10th ray
    for i in range(0, len(angles), 10):   # loop: draw rays
        ax.plot([obs[0], hits[i][0]], [obs[1], hits[i][1]],
                'b-', alpha=0.2, lw=0.5)

    # farthest point
    fa, fd, fp = farthest_point_from(obs, room)
    ax.plot([obs[0], fp[0]], [obs[1], fp[1]], 'r-', lw=2.5, label=f'farthest {fd:.1f}m')
    ax.plot(*obs, 'ko', ms=6); ax.plot(*fp, 'r*', ms=12)

    # also find farthest from corner
    corner = pts[0] + np.array([0.5, 0.5])
    fa2, fd2, fp2 = farthest_point_from(corner, room)
    ax.plot([corner[0], fp2[0]], [corner[1], fp2[1]], 'g--', lw=2,
            label=f'corner farthest {fd2:.1f}m')
    ax.plot(*corner, 'g^', ms=8)

    ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
    ax.set_title(f'{room_name}: ray cast + farthest visible point')
    ax.legend(fontsize=9)

plt.suptitle('Interior Geometry: Ray Casting & Sight Lines', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# ── summary: farthest point for each room ─────────────────────────
print('Farthest visible point from centroid in each room:')
print(f'{"Room":18}  {"Max dist m":10}  {"Min dist m":10}  {"Mean dist m"}')
print('-'*55)
for room in castle.all_rooms():                # loop over rooms
    angles, dists, hits = cast_rays(room.centroid, room, n_rays=360)
    valid = dists[dists > 0]
    if len(valid):
        print(f'{room.name:18}  {valid.max():10.2f}  {valid.min():10.2f}  {valid.mean():.2f}')

---
# §5 — Flood Simulation: 2D Shallow-Water FTCS on Room Footprint

**Shallow-water equations (linearized, FTCS finite difference):**

$$\\frac{\\partial h}{\\partial t} + H_0\\left(\\frac{\\partial u}{\\partial x} + \\frac{\\partial v}{\\partial y}\\right) = 0$$
$$\\frac{\\partial u}{\\partial t} = -g\\frac{\\partial h}{\\partial x}, \\quad \\frac{\\partial v}{\\partial t} = -g\\frac{\\partial h}{\\partial y}$$

**Boundary conditions:**  
- Walls: `u=0` (no normal flow)  
- Door (open): free-surface outflow (transmissive BC)  
- Initial condition: Gaussian pulse = flood source (e.g. burst pipe)

In [ ]:
def run_flood(room, source_pos, source_strength=0.5,
              H0=0.3, dt=0.01, n_steps=300, grid_res=0.25):
    """
    2D shallow-water FTCS flood simulation on a room polygon.
    H0 = mean water depth [m]
    Returns: h_history (n_steps, Ny, Nx), grid (x_arr, y_arr)
    """
    # build grid
    verts = np.array([v for v in room.floor.vertices])
    xmin, ymin = verts.min(0); xmax, ymax = verts.max(0)
    dx = dy = grid_res
    x_arr = np.arange(xmin, xmax+dx, dx)
    y_arr = np.arange(ymin, ymax+dy, dy)
    Nx, Ny = len(x_arr), len(y_arr)

    # mask: True = inside room
    from matplotlib.path import Path
    path = Path(np.vstack([verts, verts[0]]))
    xg, yg = np.meshgrid(x_arr, y_arr)
    pts = np.column_stack([xg.ravel(), yg.ravel()])
    inside = path.contains_points(pts).reshape(Ny, Nx)

    # fields: h (surface elevation), u,v (depth-averaged velocity)
    h = np.zeros((Ny, Nx))
    u = np.zeros((Ny, Nx))
    v = np.zeros((Ny, Nx))

    # initial condition: Gaussian flood source
    sx, sy = source_pos
    sig = 0.8   # width of source [m]
    for j in range(Ny):
        for i in range(Nx):
            r2 = (x_arr[i]-sx)**2 + (y_arr[j]-sy)**2
            h[j,i] = source_strength * np.exp(-r2/(2*sig**2)) * inside[j,i]

    g_grav = 9.81
    # CFL check
    c_wave = np.sqrt(g_grav * H0)
    CFL    = c_wave * dt / dx
    if CFL > 0.5:
        print(f'WARNING: CFL={CFL:.3f} > 0.5, unstable! Reduce dt.')

    # identify door cells (on wall boundary)
    door_mask = np.zeros((Ny, Nx), dtype=bool)
    for wall in room.walls:
        for opening in wall.openings:
            if isinstance(opening, Door):
                mp = wall.midpoint()
                # find nearest grid cell
                ix = np.argmin(np.abs(x_arr - mp[0]))
                iy = np.argmin(np.abs(y_arr - mp[1]))
                door_mask[iy, ix] = True

    h_history = [h.copy()]

    # ── FTCS time loop ──────────────────────────────────────────
    for step in range(n_steps):               # outer: time steps
        h_new = h.copy(); u_new = u.copy(); v_new = v.copy()

        for j in range(1, Ny-1):              # inner: spatial loop
            for i in range(1, Nx-1):
                if not inside[j,i]: continue
                # update h
                h_new[j,i] = h[j,i] - H0*dt/dx*(u[j,i]-u[j,i-1]) \
                                     - H0*dt/dy*(v[j,i]-v[j-1,i])
                # update u (x-momentum)
                if inside[j,i+1]:
                    u_new[j,i] = u[j,i] - g_grav*dt/dx*(h[j,i+1]-h[j,i])
                # update v (y-momentum)
                if inside[j+1,i]:
                    v_new[j,i] = v[j,i] - g_grav*dt/dy*(h[j+1,i]-h[j,i])

        # boundary: walls → zero normal velocity
        h_new[~inside] = 0; u_new[~inside] = 0; v_new[~inside] = 0

        # door: transmissive (outflow) — free gradient
        for j in range(Ny):
            for i in range(Nx):
                if door_mask[j,i]:
                    h_new[j,i] = max(0, h_new[j,i] * 0.85)  # 15% drain

        h, u, v = h_new, u_new, v_new
        if step % 30 == 0: h_history.append(h.copy())

    return np.array(h_history), x_arr, y_arr, inside, CFL


# ── Run flood on Great Hall ───────────────────────────────────────
room_flood = room_objects['Great Hall']
source     = (3.0, 2.0)   # burst pipe near corner
h_hist, x_arr, y_arr, inside, CFL = run_flood(
    room_flood, source, source_strength=0.4, H0=0.25, dt=0.008, n_steps=400
)
print(f'CFL = {CFL:.3f}, snapshots = {len(h_hist)}')

# ── Plot snapshots ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
vmax = h_hist.max()
for ax, snap in zip(axes.flat, h_hist):      # loop: snapshots
    im = ax.imshow(snap, extent=[x_arr[0],x_arr[-1],y_arr[0],y_arr[-1]],
                   origin='lower', cmap='Blues', vmin=0, vmax=vmax)
    # draw room outline
    verts = np.array([v for v in room_flood.floor.vertices])
    ax.plot(np.append(verts[:,0],verts[0,0]),
            np.append(verts[:,1],verts[0,1]), 'k-', lw=1.5)
    ax.set_aspect('equal'); ax.axis('off')
    idx = list(h_hist).index(snap) if hasattr(list(h_hist),'index') else 0

plt.colorbar(im, ax=axes[-1,-1], label='Water depth (m)')
plt.suptitle('Great Hall Flood Simulation — FTCS Shallow Water\n(Source: burst pipe at corner, door as outflow boundary)',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

# ── Total water volume vs time ────────────────────────────────────
dx = x_arr[1]-x_arr[0]
vol_vs_time = [snap.sum() * dx**2 for snap in h_hist]
time_vals   = np.arange(len(h_hist)) * 30 * 0.008

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(time_vals, vol_vs_time, 'b-', lw=2)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Total water volume (m³)')
ax.set_title('Water volume in Great Hall vs time (door draining)')
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
# §6 — Torch Room Algebra: Batch Affine Transforms, Wall Collisions, Space Metrics

**Applications:**  
- Batch transform 1000 furniture configurations (rotation + translation)  
- Detect which configurations violate wall clearances  
- Compute space utilization metrics across the whole castle

In [ ]:
# ── 1. Batch affine transforms on room vertices ────────────────────
# Represent room as a set of 2D vertex tensors
# Apply batch rotation+translation (furniture layout search)

def make_room_tensor(room):
    verts = np.array([v for v in room.floor.vertices], dtype=np.float64)
    return torch.tensor(verts, dtype=torch.float64)   # (N_verts, 2)

def batch_rotate_2d(points_b, angles_b):
    """
    points_b: (B, N, 2) — B rooms, N points each
    angles_b: (B,)      — rotation angles
    Returns:  (B, N, 2)
    """
    cos_a = torch.cos(angles_b)   # (B,)
    sin_a = torch.sin(angles_b)
    R = torch.stack([                         # (B, 2, 2)
        torch.stack([cos_a, -sin_a], dim=1),
        torch.stack([sin_a,  cos_a], dim=1),
    ], dim=1)
    return torch.bmm(points_b, R.transpose(1,2))

# ── 2. Furniture collision detection ──────────────────────────────
# A table is a 2m × 1m rectangle. Try B=2000 random placements.
B = 2000
room_t = room_objects['Great Hall']
verts_t = make_room_tensor(room_t)   # (N_v, 2)

# room bounding box
xmin_r, ymin_r = verts_t.min(0).values
xmax_r, ymax_r = verts_t.max(0).values

# random furniture positions (center of table)
torch.manual_seed(7)
tx = torch.rand(B, dtype=torch.float64) * (xmax_r - xmin_r - 2) + xmin_r + 1
ty = torch.rand(B, dtype=torch.float64) * (ymax_r - ymin_r - 2) + ymin_r + 1
theta = torch.rand(B, dtype=torch.float64) * 2 * np.pi

# table corners relative to center: 2m × 1m
table_local = torch.tensor([[-1,-0.5],[1,-0.5],[1,0.5],[-1,0.5]],
                             dtype=torch.float64)  # (4, 2)
table_batch = table_local.unsqueeze(0).expand(B, -1, -1)  # (B, 4, 2)

# rotate
table_rotated = batch_rotate_2d(table_batch, theta)        # (B, 4, 2)
# translate
centers = torch.stack([tx, ty], dim=1).unsqueeze(1)        # (B, 1, 2)
table_world = table_rotated + centers                       # (B, 4, 2)

# ── point-in-polygon test (Torch): all 4 table corners must be inside room
# Use winding number (vectorized)
def torch_point_in_polygon(points, polygon):
    """
    points:  (B, N_pts, 2)
    polygon: (N_verts, 2)
    Returns: (B, N_pts) bool
    """
    n = polygon.shape[0]
    B_p, N_pts, _ = points.shape
    inside = torch.zeros(B_p, N_pts, dtype=torch.bool)

    for i in range(n):              # loop over polygon edges
        j = (i+1) % n
        xi, yi = polygon[i,0], polygon[i,1]
        xj, yj = polygon[j,0], polygon[j,1]
        px, py = points[...,0], points[...,1]  # (B, N_pts)
        cond1 = (yi > py) != (yj > py)
        x_int = (xj - xi) * (py - yi) / (yj - yi + 1e-15) + xi
        inside ^= (cond1 & (px < x_int))
    return inside

polygon_t = verts_t  # (N_v, 2)
inside_mask = torch_point_in_polygon(table_world, polygon_t)  # (B, 4)
valid = inside_mask.all(dim=1)   # all 4 corners inside

print(f'Furniture placement: {B} random configs, '
      f'{valid.sum().item()} valid ({100*valid.float().mean().item():.1f}%)')

# ── 3. Space utilization: loop over rooms ─────────────────────────
# For each room, compute ratio of usable to gross area
# (deduct wall thickness footprint)
print('\nSpace utilization (usable / gross floor area):')
print(f'{"Room":18}  {"Gross m²":9}  {"Wall area m²":12}  {"Usable":7}  {"Util %"}')
print('-'*65)
total_gross = 0; total_usable = 0
for room in castle.all_rooms():                # loop over rooms
    wall_footprint = sum(w.length * w.thickness for w in room.walls)
    usable = room.floor_area - wall_footprint
    util   = 100 * usable / room.floor_area
    total_gross  += room.floor_area
    total_usable += usable
    print(f'{room.name:18}  {room.floor_area:9.2f}  {wall_footprint:12.2f}  {usable:7.2f}  {util:.1f}%')
print(f'{"TOTAL":18}  {total_gross:9.2f}  {"": 12}  {total_usable:7.2f}  {100*total_usable/total_gross:.1f}%')

# ── 4. Batch thermal load (Torch) ─────────────────────────────────
# Vary ΔT from 0 to 40K across all rooms
dT_vals = torch.linspace(0, 40, 200, dtype=torch.float64)

room_data = []
for room in castle.all_rooms():                # loop: precompute per room
    U_total = sum(w.area * w.material.u_value(w.thickness) for w in room.walls)
    room_data.append(U_total)

UA_tensor = torch.tensor(room_data, dtype=torch.float64).unsqueeze(1)  # (n_rooms, 1)
heat_loads = UA_tensor * dT_vals.unsqueeze(0)  # (n_rooms, 200) — broadcast

fig, ax = plt.subplots(figsize=(10, 4))
for i, room in enumerate(castle.all_rooms()):  # loop: plot each room
    ax.plot(dT_vals.numpy(), heat_loads[i].numpy() / 1000,
            lw=1.8, label=room.name)
ax.set_xlabel('ΔT (K)'); ax.set_ylabel('Heat load (kW)')
ax.set_title('Castle Ravenskeep — Thermal load vs ΔT (Torch vectorized)', fontweight='bold')
ax.legend(fontsize=8, ncol=2); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()